In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- sample_empty_df ---
class Dataset:
    def __init__(self, df, x_columns, y_columns, w_columns):
        self._df = df
        self.x_columns = x_columns
        self.y_columns = y_columns
        self.w_columns = w_columns
    def to_pandas(self):
        if isinstance(self._df, pl.DataFrame):
            return self._df.to_pandas()
        return self._df.copy()
    def to_polars(self):
        if isinstance(self._df, pd.DataFrame):
            return pl.from_pandas(self._df)
        return self._df.clone()

FIX_SAMPLE_EMPTY_DF_DATASET = Dataset(
    pd.DataFrame({"x": [1, 2, 3, 4], "y": [0, 1, 0, 1], "w": [1.0, 0.5, 1.5, 1.0]}),
    ["x"], ["y"], ["w"]
)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_sample_empty_df():
    def sample(
        ds: Dataset, n: int | None = None, frac: float | None = None, random_state: int = 42
    ) -> Dataset:
        if n == 0 or frac == 0:
            return Dataset(
                pd.DataFrame(columns=ds.to_pandas().columns),
                ds.x_columns,
                ds.y_columns,
                ds.w_columns,
            )

        df = ds.to_pandas().sample(n=n, frac=frac, random_state=random_state)
        return Dataset(df, ds.x_columns, ds.y_columns, ds.w_columns)
    return sample

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_sample_empty_df():

    def sample(
        ds: Dataset, n: int | None = None, frac: float | None = None, random_state: int = 42
    ) -> Dataset:
        if n == 0 or frac == 0:
            return Dataset(
                pl.DataFrame(schema={col: pl.Null for col in ds.to_pandas().columns}),
                ds.x_columns,
                ds.y_columns,
                ds.w_columns,
            )

        df = pl.from_pandas(ds.to_pandas()).sample(n=n, fraction=frac, seed=random_state)
        return Dataset(df, ds.x_columns, ds.y_columns, ds.w_columns)
    return sample

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: sample_empty_df ===

def _dataset_to_pl(ds):
    return ds.to_polars().sort(ds.to_polars().columns)

# L1 smoke – generated
try:
    _sample = gen_sample_empty_df()
    _r = _sample(FIX_SAMPLE_EMPTY_DF_DATASET, n=0)
    print("✅ L1 smoke gen_sample_empty_df: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_sample_empty_df: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _sample = before_sample_empty_df()
    _rb = _sample(FIX_SAMPLE_EMPTY_DF_DATASET, n=0)
    print("✅ L1 smoke before_sample_empty_df: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_sample_empty_df: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_sample_empty_df()(FIX_SAMPLE_EMPTY_DF_DATASET, n=0)
    _rg = gen_sample_empty_df()(FIX_SAMPLE_EMPTY_DF_DATASET, n=0)
    pl_assert_frame_equal(_dataset_to_pl(_rb), _dataset_to_pl(_rg), check_dtypes=False)
    print("✅ L2 equivalence sample_empty_df n=0: MATCH")
except Exception as _e:
    print(f"❌ L2 equivalence sample_empty_df: setup error — {type(_e).__name__}: {_e}")

# L3 edge – frac=0 keeps schema and no rows
try:
    _rb = before_sample_empty_df()(FIX_SAMPLE_EMPTY_DF_DATASET, frac=0)
    _rg = gen_sample_empty_df()(FIX_SAMPLE_EMPTY_DF_DATASET, frac=0)
    pl_assert_frame_equal(_dataset_to_pl(_rb), _dataset_to_pl(_rg), check_dtypes=False)
    print("✅ L3 edge sample_empty_df frac=0: MATCH")
except Exception as _e:
    print(f"❌ L3 edge sample_empty_df frac=0: {type(_e).__name__}: {_e}")

# L3 edge – positive n returns the requested number of rows
try:
    _rb = before_sample_empty_df()(FIX_SAMPLE_EMPTY_DF_DATASET, n=2, random_state=1)
    _rg = gen_sample_empty_df()(FIX_SAMPLE_EMPTY_DF_DATASET, n=2, random_state=1)
    if _rb.to_pandas().shape[0] == _rg.to_polars().height == 2:
        print("✅ L3 edge sample_empty_df n=2: MATCH row count")
    else:
        print(f"❌ L3 edge sample_empty_df n=2: MISMATCH — before={_rb.to_pandas().shape[0]}, gen={_rg.to_polars().height}")
except Exception as _e:
    print(f"❌ L3 edge sample_empty_df n=2: {type(_e).__name__}: {_e}")
